# European and American Option Pricing with Lattice Methods


In [ ]:
# Common Input Parameters

# Initial stock price
S0 = 100

# Risk-free annual interest rate (continuous compounding assumed)
r = 0.05

# Annual volatility
sigma = 0.20

# Time to maturity in years (3 months converted to years)
T = 3 / 12

# At-the-money strike price
K = S0

# Display parameters
parameters = {
    "S0": S0,
    "r": r,
    "sigma": sigma,
    "T (years)": T,
    "Strike (ATM)": K
}

parameters

{'S0': 100, 'r': 0.05, 'sigma': 0.2, 'T (years)': 0.25, 'Strike (ATM)': 100}

# European Call and Put Pricing using Binomial Tree

In [ ]:

import numpy as np
import pandas as pd

# Common parameters (from previous cell)
S0 = 100
K = 100
r = 0.05
sigma = 0.20
T = 3 / 12  # 0.25 years

# Binomial parameters
N = 200  # number of steps
dt = T / N

u = np.exp(sigma * np.sqrt(dt))
d = np.exp(-sigma * np.sqrt(dt))
p = (np.exp(r * dt) - d) / (u - d)
discount = np.exp(-r * dt)

# Stock prices at maturity
ST = np.array([S0 * (u**j) * (d**(N - j)) for j in range(N + 1)])

# Payoffs at maturity (European)
call_payoff = np.maximum(ST - K, 0)
put_payoff = np.maximum(K - ST, 0)

# Backward induction
for _ in range(N):
    call_payoff = discount * (p * call_payoff[1:] + (1 - p) * call_payoff[:-1])
    put_payoff = discount * (p * put_payoff[1:] + (1 - p) * put_payoff[:-1])

call_price = round(call_payoff[0], 2)
put_price = round(put_payoff[0], 2)

# Results table
prices_df = pd.DataFrame(
    {
        "Option Type": ["European Call", "European Put"],
        "Price": [call_price, put_price]
    }
)

prices_df

,Option Type,Price
0,European Call,4.61
1,European Put,3.37


# Delta Calculation using Binomial Tree (CRR)

In [ ]:


import numpy as np
import pandas as pd

# Common parameters
S0 = 100
K = 100
r = 0.05
sigma = 0.20
T = 3 / 12  # years
N = 200
dt = T / N

u = np.exp(sigma * np.sqrt(dt))
d = np.exp(-sigma * np.sqrt(dt))
p = (np.exp(r * dt) - d) / (u - d)
discount = np.exp(-r * dt)

# Stock prices at maturity
ST = np.array([S0 * (u**j) * (d**(N - j)) for j in range(N + 1)])

# Terminal payoffs
call_payoff = np.maximum(ST - K, 0)
put_payoff = np.maximum(K - ST, 0)

# Backward induction until time step 1
for _ in range(N - 1):
    call_payoff = discount * (p * call_payoff[1:] + (1 - p) * call_payoff[:-1])
    put_payoff = discount * (p * put_payoff[1:] + (1 - p) * put_payoff[:-1])

# Option values at time step 1
call_up, call_down = call_payoff[1], call_payoff[0]
put_up, put_down = put_payoff[1], put_payoff[0]

# Stock prices at time step 1
S_up = S0 * u
S_down = S0 * d

# Delta calculations
delta_call = round((call_up - call_down) / (S_up - S_down), 2)
delta_put = round((put_up - put_down) / (S_up - S_down), 2)

# Results table
delta_df = pd.DataFrame(
    {
        "Option Type": ["European Call", "European Put"],
        "Delta": [delta_call, delta_put]
    }
)

delta_df

,Option Type,Delta
0,European Call,0.57
1,European Put,-0.43


# Volatility Sensitivity Analysis using Binomial Tree

In [ ]:

import numpy as np
import pandas as pd

# Fixed parameters
S0 = 100
K = 100
r = 0.05
T = 3 / 12
N = 200

dt = T / N
discount = np.exp(-r * dt)

def european_option_price_binomial(sigma):
    u = np.exp(sigma * np.sqrt(dt))
    d = np.exp(-sigma * np.sqrt(dt))
    p = (np.exp(r * dt) - d) / (u - d)

    # Terminal stock prices
    ST = np.array([S0 * (u**j) * (d**(N - j)) for j in range(N + 1)])

    # Payoffs
    call = np.maximum(ST - K, 0)
    put = np.maximum(K - ST, 0)

    # Backward induction
    for _ in range(N):
        call = discount * (p * call[1:] + (1 - p) * call[:-1])
        put = discount * (p * put[1:] + (1 - p) * put[:-1])

    return round(call[0], 2), round(put[0], 2)

# Prices at 20% volatility
call_20, put_20 = european_option_price_binomial(0.20)

# Prices at 25% volatility
call_25, put_25 = european_option_price_binomial(0.25)

# Results table
vol_df = pd.DataFrame(
    {
        "Option Type": ["European Call", "European Put"],
        "Price (σ = 20%)": [call_20, put_20],
        "Price (σ = 25%)": [call_25, put_25],
        "Difference": [
            round(call_25 - call_20, 2),
            round(put_25 - put_20, 2)
        ]
    }
)

vol_df

,Option Type,Price (σ = 20%),Price (σ = 25%),Difference
0,European Call,4.61,5.59,0.98
1,European Put,3.37,4.35,0.98


# American Call Pricing using Trinomial Tree

In [ ]:


import numpy as np
import pandas as pd

# Common parameters
S0 = 100
r = 0.05
sigma = 0.20
T = 3 / 12
N = 200

dt = T / N
discount = np.exp(-r * dt)

# Trinomial parameters (recombining)
u = np.exp(sigma * np.sqrt(2 * dt))
d = 1 / u

p_u = ((np.exp(r * dt / 2) - np.exp(-sigma * np.sqrt(dt / 2))) /
       (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2

p_d = ((np.exp(sigma * np.sqrt(dt / 2)) - np.exp(r * dt / 2)) /
       (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2

p_m = 1 - p_u - p_d

strike_prices = [80, 90, 100, 110, 120]
call_prices = []

for K in strike_prices:
    # Terminal stock prices
    ST = np.array([S0 * (u ** (N - i)) for i in range(2 * N + 1)])

    # Terminal payoff
    values = np.maximum(ST - K, 0)

    # Backward induction with early exercise
    for _ in range(N):
        values = discount * (
            p_u * values[:-2] +
            p_m * values[1:-1] +
            p_d * values[2:]
        )

        ST = ST[1:-1] / u
        values = np.maximum(values, ST - K)

    call_prices.append(round(values[0], 2))

# Results table
am_call_df = pd.DataFrame({
    "Strike Price (K)": strike_prices,
    "American Call Price": call_prices
})

am_call_df

,Strike Price (K),American Call Price
0,80,21.02
1,90,11.67
2,100,4.61
3,110,1.19
4,120,0.20


# American Put Pricing using Trinomial Tree

In [ ]:

import numpy as np
import pandas as pd

# Common parameters
S0 = 100
r = 0.05
sigma = 0.20
T = 3 / 12
N = 200

dt = T / N
discount = np.exp(-r * dt)

# Trinomial parameters (recombining)
u = np.exp(sigma * np.sqrt(2 * dt))
d = 1 / u

p_u = ((np.exp(r * dt / 2) - np.exp(-sigma * np.sqrt(dt / 2))) /
       (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2

p_d = ((np.exp(sigma * np.sqrt(dt / 2)) - np.exp(r * dt / 2)) /
       (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2

p_m = 1 - p_u - p_d

strike_prices = [80, 90, 100, 110, 120]
put_prices = []

for K in strike_prices:
    # Terminal stock prices
    ST = np.array([S0 * (u ** (N - i)) for i in range(2 * N + 1)])

    # Terminal payoff (American Put)
    values = np.maximum(K - ST, 0)

    # Backward induction with early exercise
    for _ in range(N):
        values = discount * (
            p_u * values[:-2] +
            p_m * values[1:-1] +
            p_d * values[2:]
        )

        ST = ST[1:-1] / u
        values = np.maximum(values, K - ST)

    put_prices.append(round(values[0], 2))

# Results table
am_put_df = pd.DataFrame({
    "Strike Price (K)": strike_prices,
    "American Put Price": put_prices
})

am_put_df

,Strike Price (K),American Put Price
0,80,66.47
1,90,76.47
2,100,86.47
3,110,96.47
4,120,106.47


In [ ]:
# American Put Pricing using Trinomial Tree

import numpy as np
import pandas as pd

# Common parameters
S0 = 100
r = 0.05
sigma = 0.20
T = 3 / 12
N = 200

dt = T / N
discount = np.exp(-r * dt)

# Recombining trinomial parameters
u = np.exp(sigma * np.sqrt(2 * dt))
d = 1 / u

p_u = ((np.exp(r * dt / 2) - np.exp(-sigma * np.sqrt(dt / 2))) /
       (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2

p_d = ((np.exp(sigma * np.sqrt(dt / 2)) - np.exp(r * dt / 2)) /
       (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2

p_m = 1 - p_u - p_d

strike_prices = [80, 90, 100, 110, 120]
put_prices = []

for K in strike_prices:
    # Terminal stock prices
    ST = np.array([S0 * (u ** (N - i)) for i in range(2 * N + 1)])

    # Terminal payoff
    values = np.maximum(K - ST, 0)

    # Backward induction with early exercise
    for _ in range(N):
        values = discount * (
            p_u * values[:-2] +
            p_m * values[1:-1] +
            p_d * values[2:]
        )
        ST = ST[1:-1] / u
        values = np.maximum(values, K - ST)

    put_prices.append(round(values[0], 2))

# Results table
am_put_df = pd.DataFrame({
    "Strike Price (K)": strike_prices,
    "American Put Price": put_prices
})

am_put_df

,Strike Price (K),American Put Price
0,80,66.47
1,90,76.47
2,100,86.47
3,110,96.47
4,120,106.47
